# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\nPublished: {meta.datePublished}\nIdentifier: {meta.identifier}")
print(f"Citation: {meta.citeAs}")
print("\nFields available for further exploration: recordSet, field, column, etc. All references will use their `@id`.")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** We use the Croissant object model, referencing entities via their `@id`. Below, we print the available record sets and then show their fields and columns.

In [ ]:
# Explore record sets and their fields/columns

record_sets = list(meta.recordSet)  # list of RecordSet objects
print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    print(f"  name: {rs.get('name','')}")
    # Fields and columns under this record set
    
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields (by @id):")
    for field in fields:
        print(f"    - {field['@id']} - {field.get('name','')}")
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    print("  Columns (by @id):")
    for col in columns:
        print(f"    - {col['@id']} - {col.get('name','')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

> We provide an example using the first available record set.

In [ ]:
# Extract all record sets by their @id
record_sets = [rs['@id'] for rs in meta.recordSet]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records from record set {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records and columns: {list(df.columns)}\n")

# Display columns from the first record set
if record_sets:
    first_rs = record_sets[0]
    print("First 5 rows of the first record set:")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

> Choose a numeric field for demonstration (e.g., `coverage_percentage`, `peptide_count`, or `MW` if such columns are present). Use the field's column name (or `@id` if field mapping is direct in the DataFrame columns).

Update the variable `numeric_field` based on the columns displayed above.

In [ ]:
import numpy as np

record_set_id = record_sets[0]  # Using the first available
df = dataframes[record_set_id]

# Choose a numeric field from the available columns.
# If the columns include 'MW', 'peptide_count', 'coverage_percentage', use that. Otherwise, set to an actual numeric column.
candidate_num_fields = [c for c in df.columns if df[c].dtype in [np.float64, np.int64, float, int] or pd.api.types.is_numeric_dtype(df[c])]
if not candidate_num_fields:
    candidate_num_fields = [c for c in df.columns if 'count' in c.lower() or 'mw' in c.lower() or 'coverage' in c.lower()]
if not candidate_num_fields:
    print("No obvious numeric fields available for demonstration.")
else:
    numeric_field = candidate_num_fields[0]

    # Remove NA values in analysis
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    valid_df = df.dropna(subset=[numeric_field])
    threshold = valid_df[numeric_field].median()  # Example threshold: median
    filtered_df = valid_df[valid_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping - pick a categorical field if available
    group_fields = [c for c in df.columns if df[c].dtype == object and c != numeric_field]
    group_field = group_fields[0] if group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped (mean) of {numeric_field} by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

> Below we show one example: the distribution of the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt

if candidate_num_fields:
    field = numeric_field
    plt.figure(figsize=(8,4))
    df[field].dropna().hist(bins=30)
    plt.xlabel(field)
    plt.ylabel('Count')
    plt.title(f"Distribution of {field}")
    plt.show()
    
    if group_field is not None:
        plt.figure(figsize=(10,5))
        filtered_df.boxplot(column=field, by=group_field, rot=60)
        plt.title(f"Boxplot of {field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(field)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to:
1. Load the metadata and records from the FAIR² Croissant-based mass spectrometry dataset using `mlcroissant`;
2. Explore available record sets, fields, and columns by `@id`;
3. Load data into Pandas, filter and normalize numeric fields, and perform group-wise analysis;
4. Visualize data distributions to gain insights.

**Tips:**
- For further analysis, consult the field and column `@id`s to access the precise biological quantities or parameters desired.
- Extend the EDA and visualizations to more record sets and fields for deeper insights relevant to your research.